# Lesson 05: Vector Stores vs. Vector Databases

This notebook explores the architectural and operational differences between lightweight embedded **Vector Stores** (e.g., FAISS, ChromaDB) and enterprise **Vector Databases** (e.g., Pinecone, Qdrant, Milvus):
1. **In-Memory Vector Store Operations** (Sub-millisecond retrieval benchmark).
2. **CRUD & Lifecycle Analysis** (Insert, Search, and Update/Delete trade-offs).
3. **Metadata Filtering** (Pre-filtering vs. Post-filtering).
4. **Engineering Decision Engine** (Rule-based recommendation utility for RAG architectures).

> For detailed architectural breakdowns, comparison matrices, and technology catalogs, see the [Lesson Documentation](README.md).

## 1. Setup & Environment
Initialize dependencies and remote embeddings.

In [ ]:
import os
import time
import numpy as np
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

load_dotenv()

# Initialize embedding model (OpenAI if key exists, otherwise remote HuggingFace)
openai_key = os.getenv('OPENAI_API_KEY')
if openai_key and len(openai_key) > 10:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    print('Using OpenAI Embeddings (text-embedding-3-small)')
else:
    from langchain_huggingface import HuggingFaceEndpointEmbeddings
    embeddings = HuggingFaceEndpointEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')
    print('Using Hugging Face Endpoint Embeddings (all-MiniLM-L6-v2)')


## 2. Vector Store In-Memory Performance Benchmark
Vector stores run within application process memory (RAM), achieving microsecond-level query latencies.

In [ ]:
# Sample corpus representing documents across domains
sample_corpus = [
    Document(page_content='LangChain provides standard interfaces for chains, agents, and retrieval.', metadata={'category': 'ai', 'tier': 'free'}),
    Document(page_content='FAISS is an in-memory library for dense vector similarity search developed by Meta.', metadata={'category': 'database', 'tier': 'open_source'}),
    Document(page_content='Pinecone is a managed cloud-native vector database designed for high availability at scale.', metadata={'category': 'database', 'tier': 'enterprise'}),
    Document(page_content='PostgreSQL supports vector similarity search via the pgvector extension.', metadata={'category': 'database', 'tier': 'open_source'}),
    Document(page_content='Qdrant is an open-source vector database written in Rust with advanced filtering.', metadata={'category': 'database', 'tier': 'open_source'}),
]

# Build in-memory FAISS Vector Store
vector_store = FAISS.from_documents(sample_corpus, embeddings)
print(f"Vector Store initialized with {vector_store.index.ntotal} vectors in memory.")

# Benchmark query latency
query = 'What is an enterprise cloud vector database?'
start_time = time.perf_counter()
results = vector_store.similarity_search(query, k=2)
latency_ms = (time.perf_counter() - start_time) * 1000

print(f"\nSearch completed in: {latency_ms:.3f} ms")
for rank, doc in enumerate(results, 1):
    print(f"  #{rank}: {doc.page_content} (Category: {doc.metadata['category']})")


## 3. Metadata Filtering in Vector Stores vs. Vector Databases
Demonstrate how metadata tags enable targeted retrieval.

In [ ]:
# Filter query: only retrieve documents where tier == 'enterprise'
enterprise_results = vector_store.similarity_search(
    'database options',
    k=2,
    filter={'tier': 'enterprise'}
)

print("Filtered Search Results (tier == 'enterprise'):")
for doc in enterprise_results:
    print(f"  Content: {doc.page_content}")
    print(f"  Metadata: {doc.metadata}\n")


## 4. Architectural Decision Engine
A decision utility to guide technology selection based on workload requirements.

In [ ]:
def recommend_vector_storage(
    num_vectors: int,
    is_production: bool,
    needs_realtime_crud: bool,
    budget_usd_monthly: float
) -> dict:
    """Determine optimal vector storage architecture based on engineering constraints."""
    if not is_production or num_vectors < 50000:
        return {
            'Type': 'Vector Store (Embedded)',
            'Recommendation': 'FAISS or ChromaDB',
            'Rationale': 'Dataset fits comfortably in memory. Zero cost, microsecond latency, zero infrastructure overhead.'
        }
    
    if needs_realtime_crud and num_vectors >= 1000000:
        if budget_usd_monthly >= 70:
            return {
                'Type': 'Vector Database (Managed Cloud)',
                'Recommendation': 'Pinecone or Qdrant Cloud',
                'Rationale': 'Full real-time CRUD, automatic horizontal scaling, high availability SLA, zero maintenance.'
            }
        else:
            return {
                'Type': 'Vector Database (Self-Hosted)',
                'Recommendation': 'Qdrant (Docker) or Milvus / Pgvector',
                'Rationale': 'Open-source enterprise DBMS. Full CRUD and scale without managed service recurring costs.'
            }
            
    return {
        'Type': 'Hybrid / Embedded Persistent',
        'Recommendation': 'ChromaDB Persistent or Pgvector',
        'Rationale': 'Offers local persistence and simple updates without dedicated distributed cluster complexity.'
    }

# Test decision engine across scenarios
scenarios = [
    ('Prototype / Academic RAG', 5000, False, False, 0),
    ('Production Knowledge Base (100K docs)', 100000, True, True, 20),
    ('Enterprise SaaS (10M real-time vectors)', 10000000, True, True, 300),
]

for name, count, prod, crud, budget in scenarios:
    rec = recommend_vector_storage(count, prod, crud, budget)
    print(f"Scenario: {name}")
    print(f"  Category:       {rec['Type']}")
    print(f"  Recommendation: {rec['Recommendation']}")
    print(f"  Rationale:      {rec['Rationale']}\n")
